# Lezione 1 · Il neurone e i pesi

Il neurone della serata: tre fatti, tre pesi, un'inclinazione e una vela. Stasera vado ad allenarmi?

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def softmax(z):
    e = np.exp(np.array(z, dtype=float) - np.max(z))
    return e / e.sum()

## Il neurone in una formula

*Nel racconto: i sassolini, i segni di china, il cerchio aperto in alto e il segno che ne esce.*

I fatti sono energia (da 0 a 1), pioggia (0 o 1) e compagno (0 o 1). I pesi sono quelli del foglio del Maestro: 2,0 per l'energia, −1,5 per la pioggia, 0,1 per il compagno; l'inclinazione, il bias, è −0,5.

In [2]:
w = np.array([2.0, -1.5, 0.1])
b = -0.5

def spinta(x, w=w, b=b):
    return float(np.dot(w, x)) + b

serata = np.array([0.8, 1, 1])
z = spinta(serata)
controlla("la spinta della serata, z", z, -0.3)

# il conto sbagliato dell'allievo: la pioggia sommata come se spingesse
controlla("il conto sbagliato dell'allievo", 0.8 * 2 + 1.5 + 0.1 - 0.5, 2.7)

✓  la spinta della serata, z: calcolato -0.3, nel libro -0.3
✓  il conto sbagliato dell'allievo: calcolato 2.7, nel libro 2.7


## Le tre attivazioni

*Nel racconto: la tavoletta, la vela, la scogliera.*

In [3]:
gradino = lambda z: 1 if z > 0 else 0
relu = lambda z: max(0.0, z)

conta("gradino(−0,3): la tavoletta resta intera", gradino(z), 0)
controlla("e^0,3", math.exp(0.3), 1.350, 0.001)
controlla("sigmoide(−0,3): la vela al 43%", sigmoide(z), 0.426, 0.001)
controlla("e^−1,2", math.exp(-1.2), 0.301, 0.001)
controlla("senza pioggia: la spinta", spinta([0.8, 0, 1]), 1.2)
controlla("sigmoide(1,2): la vela al 77%", sigmoide(1.2), 0.769, 0.001)
controlla("ReLU(−0,3): il mare resta sotto il bordo", relu(z), 0)
controlla("ReLU(1,2)", relu(1.2), 1.2)

# l'allievo entusiasta: stessi pesi, inclinazione +1, sotto la pioggia
controlla("l'entusiasta sotto la pioggia: la spinta", spinta(serata, b=1.0), 1.2)

# la pendenza della vela
pendenza = lambda z: sigmoide(z) * (1 - sigmoide(z))
controlla("pendenza della sigmoide a metà albero", pendenza(0), 0.25)
controlla("pendenza della sigmoide in −0,3", pendenza(-0.3), 0.245, 0.001)

✓  gradino(−0,3): la tavoletta resta intera: calcolato 0, nel libro 0
✓  e^0,3: calcolato 1.35, nel libro 1.35
✓  sigmoide(−0,3): la vela al 43%: calcolato 0.4256, nel libro 0.426
✓  e^−1,2: calcolato 0.3012, nel libro 0.301
✓  senza pioggia: la spinta: calcolato 1.2, nel libro 1.2
✓  sigmoide(1,2): la vela al 77%: calcolato 0.7685, nel libro 0.769
✓  ReLU(−0,3): il mare resta sotto il bordo: calcolato 0, nel libro 0
✓  ReLU(1,2): calcolato 1.2, nel libro 1.2
✓  l'entusiasta sotto la pioggia: la spinta: calcolato 1.2, nel libro 1.2
✓  pendenza della sigmoide a metà albero: calcolato 0.25, nel libro 0.25
✓  pendenza della sigmoide in −0,3: calcolato 0.2445, nel libro 0.245


## La softmax

*Nel racconto: dojo, parco o divano.*

In [4]:
punteggi = [2.0, 1.0, 0.1]
esp = np.exp(punteggi)
for nome, v, atteso in zip(["e^2", "e^1", "e^0,1"], esp, [7.39, 2.72, 1.11]):
    controlla(nome, v, atteso)
controlla("somma", esp.sum(), 11.21)
p = softmax(punteggi)
for nome, v, atteso in zip(["dojo", "parco", "divano"], p, [0.66, 0.24, 0.10]):
    controlla(nome, v, atteso)
controlla("dojo / parco = e", p[0] / p[1], 2.72)
# con due sole opzioni la softmax è la sigmoide della differenza
controlla("softmax a due = sigmoide della differenza", softmax([2.0, 1.0])[0] - sigmoide(1.0), 0, 1e-12)

✓  e^2: calcolato 7.389, nel libro 7.39
✓  e^1: calcolato 2.718, nel libro 2.72
✓  e^0,1: calcolato 1.105, nel libro 1.11
✓  somma: calcolato 11.21, nel libro 11.21
✓  dojo: calcolato 0.659, nel libro 0.66
✓  parco: calcolato 0.2424, nel libro 0.24
✓  divano: calcolato 0.09857, nel libro 0.1
✓  dojo / parco = e: calcolato 2.718, nel libro 2.72
✓  softmax a due = sigmoide della differenza: calcolato 0, nel libro 0


## Il prodotto scalare

*Nel racconto: la squadra per la gara, il profilo del Maestro e i tre candidati.*

In [5]:
profilo = np.array([2, 1, -1.5])          # velocità, forza, impazienza
candidati = {"Aiko": [0.9, 0.3, 0.1], "Kenta": [0.2, 0.9, 0.8], "Hiro": [0.1, 0.2, 1.0]}
for (nome, x), atteso in zip(candidati.items(), [1.95, 0.1, -1.1]):
    controlla(f"punteggio di {nome}", np.dot(profilo, x), atteso)

controlla("nella serata, w·x", np.dot(w, serata), 0.2)

# l'insidia: Sora, forte in tutto ma un po' impaziente
sora = np.array([1, 1, 0.5])
controlla("punteggio di Sora", np.dot(profilo, sora), 2.25)

lung = np.linalg.norm
coseno = lambda a, c: np.dot(a, c) / (lung(a) * lung(c))
controlla("lunghezza del profilo", lung(profilo), 2.69)
controlla("lunghezza di Aiko", lung(candidati["Aiko"]), 0.95)
controlla("coseno di Aiko", coseno(profilo, candidati["Aiko"]), 0.76)
controlla("lunghezza di Sora", lung(sora), 1.5)
controlla("coseno di Sora", coseno(profilo, sora), 0.56)

# elemento per elemento (Hadamard): non somma
print("w ⊙ x per la serata:", w * serata, "(nel libro 1,6; −1,5; 0,1)")

✓  punteggio di Aiko: calcolato 1.95, nel libro 1.95
✓  punteggio di Kenta: calcolato 0.1, nel libro 0.1
✓  punteggio di Hiro: calcolato -1.1, nel libro -1.1
✓  nella serata, w·x: calcolato 0.2, nel libro 0.2
✓  punteggio di Sora: calcolato 2.25, nel libro 2.25
✓  lunghezza del profilo: calcolato 2.693, nel libro 2.69
✓  lunghezza di Aiko: calcolato 0.9539, nel libro 0.95
✓  coseno di Aiko: calcolato 0.7592, nel libro 0.76
✓  lunghezza di Sora: calcolato 1.5, nel libro 1.5
✓  coseno di Sora: calcolato 0.5571, nel libro 0.56
w ⊙ x per la serata: [ 1.6 -1.5  0.1] (nel libro 1,6; −1,5; 0,1)


## L'iperpiano

*Nel racconto: la linea nella sabbia, il taglio dell'anguria.*

Con energia e pioggia soltanto, la linea è 2,0·x₁ − 1,5·x₂ − 0,5 = 0, cioè x₁ = 0,25 + 0,75·x₂.

In [6]:
linea = lambda pioggia, b=-0.5: (1.5 * pioggia - b) / 2.0     # l'energia in cui la spinta è zero
controlla("senza pioggia la linea passa per", linea(0), 0.25)
controlla("con la pioggia la linea passa per", linea(1), 1.0)
controlla("distanza del confine dall'origine", abs(-0.5) / math.hypot(2.0, 1.5), 0.2)
controlla("la linea dell'entusiasta (b = +1), sotto la pioggia", linea(1, b=1.0), 0.25)

✓  senza pioggia la linea passa per: calcolato 0.25, nel libro 0.25
✓  con la pioggia la linea passa per: calcolato 1, nel libro 1
✓  distanza del confine dall'origine: calcolato 0.2, nel libro 0.2
✓  la linea dell'entusiasta (b = +1), sotto la pioggia: calcolato 0.25, nel libro 0.25


## Regressione lineare e logistica

*Nel racconto: "era statistica?".*

In [7]:
controlla("i piegamenti di Kenta dopo 6 settimane", 10 + 2 * 6, 22)
controlla("sbagliare di 13 costa", 13 ** 2, 169)
controlla("sbagliare di 3 costa", 3 ** 2, 9)
print(f"   169 / 9 = {169 / 9:.1f}: «quasi venti volte meno»")
controlla("la retta a 400 ore", (400 - 10) / (200 - 10), 2.05)

✓  i piegamenti di Kenta dopo 6 settimane: calcolato 22, nel libro 22
✓  sbagliare di 13 costa: calcolato 169, nel libro 169
✓  sbagliare di 3 costa: calcolato 9, nel libro 9
   169 / 9 = 18.8: «quasi venti volte meno»
✓  la retta a 400 ore: calcolato 2.053, nel libro 2.05


## La cross-entropy

*Nel racconto: l'esame di primo dan, "quanto sei sicuro di romperla?".*

In [8]:
for detto, atteso in [(0.99, 0.01), (0.01, 4.6), (0.6, 0.51), (0.4, 0.92), (0.5, 0.69)]:
    controlla(f"punti persi avendo dato {detto:.0%} a ciò che è successo", -math.log(detto), atteso)
controlla("il neurone della serata: aveva detto 43%, e sei andato", -math.log(0.43), 0.84)

# un neurone sicurissimo e sbagliato: p = 0,01, y = 1
p_, y_ = 0.01, 1
controlla("correzione con la cross-entropy, |p − y|", abs(p_ - y_), 0.99)
controlla("correzione con l'errore quadratico", abs(p_ - y_) * p_ * (1 - p_), 0.0098, 0.0001)
print(f"   rapporto: {1 / (p_ * (1 - p_)):.0f} volte («cento volte meno»)")

✓  punti persi avendo dato 99% a ciò che è successo: calcolato 0.01005, nel libro 0.01
✓  punti persi avendo dato 1% a ciò che è successo: calcolato 4.605, nel libro 4.6
✓  punti persi avendo dato 60% a ciò che è successo: calcolato 0.5108, nel libro 0.51
✓  punti persi avendo dato 40% a ciò che è successo: calcolato 0.9163, nel libro 0.92
✓  punti persi avendo dato 50% a ciò che è successo: calcolato 0.6931, nel libro 0.69
✓  il neurone della serata: aveva detto 43%, e sei andato: calcolato 0.844, nel libro 0.84
✓  correzione con la cross-entropy, |p − y|: calcolato 0.99, nel libro 0.99
✓  correzione con l'errore quadratico: calcolato 0.009801, nel libro 0.0098
   rapporto: 101 volte («cento volte meno»)


## La correzione del diario

*Nel racconto: la serata in cui il neurone aveva detto 43%, e invece sei andato.*

Come nel libro, p è arrotondato a 0,43 e il learning rate è η = 0,5.

In [9]:
p, y, eta = 0.43, 1, 0.5
fattore = -eta * (p - y)
controlla("il fattore comune", fattore, 0.285, 0.0005)
w_nuovo = w + fattore * serata
b_nuovo = b + fattore
for nome, v, conti, tabella in zip(["energia", "pioggia", "compagno"], w_nuovo, [2.228, -1.215, 0.385], [2.23, -1.22, 0.39]):
    controlla(f"peso {nome}, nei conti", v, conti, 0.0005)
    controlla(f"peso {nome}, nella tabella del racconto", v, tabella, 0.006)
controlla("inclinazione, nei conti", b_nuovo, -0.215, 0.0005)
controlla("inclinazione, nella tabella del racconto", b_nuovo, -0.22, 0.006)
z_nuovo = spinta(serata, w_nuovo, b_nuovo)
controlla("la spinta con i pesi nuovi", z_nuovo, 0.74)
controlla("che cosa dice il neurone dopo una correzione", sigmoide(z_nuovo), 0.68)

# generalizzare: una serata che nel diario non c'è mai stata
controlla("energia 0,55, pioggia, nessun compagno", sigmoide(spinta([0.55, 1, 0])), 0.29)

✓  il fattore comune: calcolato 0.285, nel libro 0.285
✓  peso energia, nei conti: calcolato 2.228, nel libro 2.228
✓  peso energia, nella tabella del racconto: calcolato 2.228, nel libro 2.23
✓  peso pioggia, nei conti: calcolato -1.215, nel libro -1.215
✓  peso pioggia, nella tabella del racconto: calcolato -1.215, nel libro -1.22
✓  peso compagno, nei conti: calcolato 0.385, nel libro 0.385
✓  peso compagno, nella tabella del racconto: calcolato 0.385, nel libro 0.39
✓  inclinazione, nei conti: calcolato -0.215, nel libro -0.215
✓  inclinazione, nella tabella del racconto: calcolato -0.215, nel libro -0.22
✓  la spinta con i pesi nuovi: calcolato 0.7374, nel libro 0.74
✓  che cosa dice il neurone dopo una correzione: calcolato 0.6764, nel libro 0.68
✓  energia 0,55, pioggia, nessun compagno: calcolato 0.2891, nel libro 0.29


## Preparare i fatti

*Nel racconto: il sasso grosso come un pugno accanto ai sassolini.*

In [10]:
controlla("8,5 ore di sonno, con media 7 e deviazione 1,5", (8.5 - 7) / 1.5, 1)
giorni = ["lunedì", "martedì", "mercoledì", "giovedì", "venerdì", "sabato", "domenica"]
one_hot = [1 if g == "mercoledì" else 0 for g in giorni]
print("mercoledì:", one_hot)
assert one_hot == [0, 0, 1, 0, 0, 0, 0]

✓  8,5 ore di sonno, con media 7 e deviazione 1,5: calcolato 1, nel libro 1
mercoledì: [0, 0, 1, 0, 0, 0, 0]


## I parametri del giornale

*Nel racconto: il titolo "da 70 miliardi di parametri".*

In [11]:
parametri = 70e9
controlla("a 16 bit (GB)", parametri * 2 / 1e9, 140)
controlla("a 4 bit (GB)", parametri * 0.5 / 1e9, 35)
conta("il nostro neurone, 4 parametri a 16 bit (byte)", 4 * 2, 8)

✓  a 16 bit (GB): calcolato 140, nel libro 140
✓  a 4 bit (GB): calcolato 35, nel libro 35
✓  il nostro neurone, 4 parametri a 16 bit (byte): calcolato 8, nel libro 8


## Kata per casa

**Esercizio 1.** La serata (0,6; 0; 0): energia 0,6, niente pioggia, nessun compagno.

In [12]:
z1 = spinta([0.6, 0, 0])
controlla("spinta", z1, 0.7)
controlla("e^−0,7", math.exp(-0.7), 0.497, 0.001)
controlla("probabilità", sigmoide(z1), 0.67)

✓  spinta: calcolato 0.7, nel libro 0.7
✓  e^−0,7: calcolato 0.4966, nel libro 0.497
✓  probabilità: calcolato 0.6682, nel libro 0.67


**Esercizio 2.** Il punteggio di Yumi è il prodotto scalare tra il profilo (2; 1; −1,5) e le sue qualità (0,5; 0,5; 0,5).

In [13]:
yumi = np.dot(profilo, [0.5, 0.5, 0.5])
controlla("punteggio di Yumi", yumi, 0.75)
assert 0.1 < yumi < 1.95      # più di Kenta, meno di Aiko

✓  punteggio di Yumi: calcolato 0.75, nel libro 0.75


## Prova tu

Cambia i pesi e guarda come cambia la decisione. Che peso dovrebbe avere la pioggia perché, con energia piena, tu resti comunque a casa?

In [14]:
for energia in [0.2, 0.5, 0.8, 1.0]:
    for piove in [0, 1]:
        print(f"energia {energia}, piove {piove}: {sigmoide(spinta([energia, piove, 1])):.0%}")

energia 0.2, piove 0: 50%
energia 0.2, piove 1: 18%
energia 0.5, piove 0: 65%
energia 0.5, piove 1: 29%
energia 0.8, piove 0: 77%
energia 0.8, piove 1: 43%
energia 1.0, piove 0: 83%
energia 1.0, piove 1: 52%
